In [ ]:
# 套件區域
# ==========================================
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import warnings
import math
import os
import sqlite3
import logging
import time
import json

# 忽略 pandas 和 statsmodels 的警告以保持輸出整潔
warnings.filterwarnings('ignore')

In [ ]:
# 參數區域
# ==========================================
FAST_TEST_MODE = True
if FAST_TEST_MODE:
    print("【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。")
    # 縮短時間：涵蓋 2020 疫情崩盤與 2022 升息的壓力測試區間
    START_DATE = '2019-01-01'
    END_DATE = '2022-12-31'
    # 縮小股票池：僅測試單一板塊，運算量大幅減少
    TARGET_SECTOR = None  
else:
    print("【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。")
    # 論文要求的全樣本時間
    START_DATE = '2000-01-01'
    END_DATE = '2025-12-31'
    # 測試全市場 (設為 None 代表不限制單一產業)
    TARGET_SECTOR = None  

# 資料庫配置
DB_PATH = r'..\data\sp500.db'
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv'
USE_DYNAMIC_SECTORS = True  # 是否使用動態產業補齊

# 視窗參數
FORMATION_WINDOW = 252    # 約 1 年
TRADING_WINDOW = 126      # 約 6 個月
ROLLING_WINDOW = 21       # 約 1 個月 (梯隊步長)
MIN_HISTORY_DAYS = 200    # 最少歷史資料

# 配對篩選參數
TOP_N_PAIRS = 10          # 每個視窗選出前 N 組配對
P_THRESHOLD = 0.01

# 交易參數
Z_ENTRY = 2.0               # 進場標準差閾值（SSD 方法中改用 2×σ_formation）
Z_EXIT = 0.0                # 平倉標準差閾值
TRANSACTION_COST = 0.0029   # 單邊交易成本 (0.29%)
PAIR_MAX_LOSS_PCT = 0.15         # 停損 (0 = 無停損)

# 資金配置
INITIAL_CAPITAL = 10000  # 初始本金
CAPITAL_TRANCHES = math.ceil(TRADING_WINDOW / ROLLING_WINDOW)

# 確保路徑存在
os.makedirs(os.path.dirname(IMPUTED_SECTOR_PATH), exist_ok=True)

print(f"""
【回測參數設置】
時間期間: {START_DATE} ~ {END_DATE}
形成期: {FORMATION_WINDOW} 天
交易期: {TRADING_WINDOW} 天
步長: {ROLLING_WINDOW} 天
配對數: {TOP_N_PAIRS}
初始資金: ${INITIAL_CAPITAL:,.0f}
交易成本: {TRANSACTION_COST*100:.2f}%
""")

In [ ]:
# ==========================================
class FetchData:
    def __init__(self, db_path, start_date, end_date, target_sector, use_dynamic, min_days, imputed_path=IMPUTED_SECTOR_PATH):
        self.db_path = db_path
        self.start_date = start_date
        self.end_date = end_date
        self.target_sector = target_sector
        self.use_dynamic = use_dynamic
        self.imputed_path = imputed_path
        self.min_days = min_days
        
        # 確保快取路徑存在
        os.makedirs(os.path.dirname(self.imputed_path), exist_ok=True)

    def fix_unknown_sectors(self, sector_df):
        """具備本機快取與全域開關控制的產業補齊"""
        if not self.use_dynamic:
            print("不使用動態產業補齊。")
            return sector_df

        if os.path.exists(self.imputed_path):
            print(f"從本機快取載入已補齊的產業分類: {self.imputed_path}")
            cached_df = pd.read_csv(self.imputed_path)
            update_df = cached_df.set_index('ticker')
            sector_df = sector_df.set_index('ticker')
            sector_df.update(update_df)
            return sector_df.reset_index()

        unknown_mask = sector_df['sector'] == 'Unknown'
        unknown_tickers = sector_df[unknown_mask]['ticker'].tolist()
        
        if not unknown_tickers:
            return sector_df

        print(f"找不到本機快取，正在透過 API 補齊 {len(unknown_tickers)} 檔股票的產業分類...")
        yf_logger = logging.getLogger('yfinance')
        original_level = yf_logger.level
        yf_logger.setLevel(logging.CRITICAL) 
        
        fixed_sectors = []
        for i, ticker in enumerate(unknown_tickers):
            try:
                info = yf.Ticker(ticker).info
                sector = info.get('sector', 'Unknown')
                fixed_sectors.append({'ticker': ticker, 'sector': sector})
                time.sleep(0.02) 
            except Exception:
                fixed_sectors.append({'ticker': ticker, 'sector': 'Unknown'})
                
            if (i + 1) % 50 == 0:
                print(f"已處理 {i + 1} / {len(unknown_tickers)}...")
                
        yf_logger.setLevel(original_level)
        
        fetched_df = pd.DataFrame(fixed_sectors)
        fetched_df.to_csv(self.imputed_path, index=False)
        print(f"API 抓取完畢！已將動態產業分類永久儲存至: {self.imputed_path}")
        
        update_df = fetched_df.set_index('ticker')
        sector_df = sector_df.set_index('ticker')
        sector_df.update(update_df)
        sector_df = sector_df.reset_index()
        
        remaining = len(sector_df[sector_df['sector'] == 'Unknown'])
        print(f"補齊完成！剩餘真實無法識別(已下市)的股票數量: {remaining}")
        return sector_df

    def preprocess_prices(self, prices_df, sector_df):
        """數據預處理：樞紐、前向填充、去除稀疏股票與 Unknown 產業"""
        pivot = prices_df.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
        pivot.index = pd.to_datetime(pivot.index)
        pivot.sort_index(inplace=True)
        
        # 前向填充（最多 5 天）
        pivot.ffill(limit=5, inplace=True)
        
        # 保留有足夠歷史資料的股票
        valid = pivot.columns[pivot.notna().sum() >= self.min_days]
        pivot = pivot[valid]
        
        # 篩選掉 Sector 為 'Unknown' 的股票
        known_tickers = sector_df[sector_df['sector'] != 'Unknown']['ticker'].tolist()
        final_valid = [col for col in pivot.columns if col in known_tickers]
        pivot = pivot[final_valid]
        
        print(f'✓ 數據矩陣：{len(pivot)} 天 × {len(pivot.columns)} 檔股票')
        print(f'✓ 時間範圍：{pivot.index[0].date()} ~ {pivot.index[-1].date()}')
        
        return pivot

    def fetch_all(self):
        """整合：讀取產業、補齊產業、生成 SQL 條件篩選價量、數據預處理"""
        if not os.path.exists(self.db_path):
            self._create_dummy_db()

        conn = sqlite3.connect(self.db_path)
        
        # 1. 優先加載行業分類
        print("\n從資料庫加載行業數據...")
        sector_queries = [
            "SELECT ticker, sector FROM tickers",
            "SELECT ticker, sector FROM sp500_components GROUP BY ticker",
        ]
        
        sector_df = None
        for q in sector_queries:
            try:
                sector_df = pd.read_sql_query(q, conn)
                if len(sector_df) > 0:
                    print(f'✓ 加載行業數據：{len(sector_df):,} 檔股票')
                    break
            except Exception:
                continue
                
        if sector_df is None or len(sector_df) == 0:
            print('⚠️ 未找到行業表，建立空的 DataFrame。')
            sector_df = pd.DataFrame(columns=['ticker', 'sector'])
            
        # 2. 動態補齊產業
        sector_df = self.fix_unknown_sectors(sector_df)
        
        # 3. 處理 Target Sector 並組裝 SQL IN 條件
        ticker_filter_sql = ""
        if self.target_sector is not None:
            target_tickers = sector_df[sector_df['sector'] == self.target_sector]['ticker'].tolist()
            print(f"🎯 板塊過濾：將在 SQL 階段嚴格篩選 {self.target_sector} 板塊之股票，共 {len(target_tickers)} 檔")
            
            if not target_tickers:
                conn.close()
                raise ValueError(f"資料庫中找不到屬於 {self.target_sector} 的股票！")
            
            # 建立 SQL 所需的 IN 字串格式
            if len(target_tickers) == 1:
                ticker_filter_sql = f" AND ticker = '{target_tickers[0]}'"
            else:
                ticker_filter_sql = f" AND ticker IN {tuple(target_tickers)}"
            
            # 過濾 DataFrame 以便後續匹配
            sector_df = sector_df[sector_df['sector'] == self.target_sector]
        else:
            print("🌍 全市場模式：將加載所有符合時間條件的股票")

        # 4. 在 SQL 階段即加上 ticker_filter_sql 篩選，降低記憶體浪費
        print("\n從資料庫加載價格數據...")
        price_queries = [
            (f"SELECT date, ticker, adj_close AS close FROM daily_prices "
             f"WHERE date BETWEEN '{self.start_date}' AND '{self.end_date}'{ticker_filter_sql} ORDER BY date"),
            (f"SELECT date, ticker, close FROM stock_prices "
             f"WHERE date BETWEEN '{self.start_date}' AND '{self.end_date}'{ticker_filter_sql} ORDER BY date"),
        ]
        
        prices_df = None
        for q in price_queries:
            try:
                prices_df = pd.read_sql_query(q, conn, parse_dates=['date'])
                if len(prices_df) > 0:
                    print(f'✓ 加載價格數據：{len(prices_df):,} 筆記錄')
                    break
            except Exception:
                continue
        
        conn.close()
        
        if prices_df is None or len(prices_df) == 0:
            raise RuntimeError("無法從資料庫加載價格數據，請確認資料表名稱與欄位。")

        # 5. 數據預處理 (樞紐、前向填充、剃除 Unknown)
        print("\n預處理價格數據...")
        price_pivot = self.preprocess_prices(prices_df, sector_df)
        
        # 6. 徹底清理 Unknown 並建立行業對應字典
        valid_tickers = price_pivot.columns.tolist()
        # 確保完全排除 Unknown，且只加入有價格數據的有效 ticker
        sector_df_valid = sector_df[sector_df['sector'] != 'Unknown']
        sector_map = sector_df_valid[sector_df_valid['ticker'].isin(valid_tickers)].set_index('ticker')['sector'].to_dict()
        
        return price_pivot, sector_map

In [ ]:
# 策略模型 2：最小距離法 (SSD)
# ==========================================
class SSDPairsTrading:
    def __init__(self, data, sector_map, formation_days, trading_days, rolling_window, capital_tranches, initial_capital, z_entry, z_exit, top_n_pairs, fee, pair_max_loss_pct):
        self.data = data
        self.sector_map = sector_map
        self.formation_days = formation_days
        self.trading_days = trading_days
        self.rolling_window = rolling_window
        self.capital_tranches = capital_tranches
        self.initial_capital = initial_capital
        self.top_n_pairs = top_n_pairs
        self.entry = z_entry
        self.exit = z_exit
        self.fee = fee
        self.pair_max_loss_pct = pair_max_loss_pct
        self.portfolio_returns = []
        self.cumulative_returns = []
        self.history_pairs = []

    def find_best_pairs(self, formation_data):
        norm_base = formation_data.iloc[0]
        norm_data = formation_data / norm_base
        
        distances = []
        valid_tickers = norm_data.columns.tolist()

        for stock_a, stock_b in combinations(valid_tickers, 2):
            spread = norm_data[stock_a] - norm_data[stock_b]
            std = spread.std()
            
            # 【修正】防呆機制：若兩檔股票價格走勢完全一樣 (如 GOOG/GOOGL)，剔除之
            if std < 1e-4: continue 
                
            sector_a = self.sector_map.get(stock_a, 'Unknown')
            sector_b = self.sector_map.get(stock_b, 'Unknown')
            
            distances.append({
                'Pair': (stock_a, stock_b),
                'Sector': sector_a if sector_a == sector_b else f"{sector_a}/{sector_b}",
                'SSD': np.sum(spread**2),
                'Std': std
            })

        if not distances: return pd.DataFrame()
        return pd.DataFrame(distances).nsmallest(self.top_n_pairs, 'SSD').reset_index(drop=True)

    def trade_pairs(self, trading_data, selected_pairs, window_capital):
        period_pnl = np.zeros(len(trading_data))
        if selected_pairs.empty: return period_pnl

        capital_per_pair = int(window_capital // len(selected_pairs))
        
        pair_params = []
        for _, row in selected_pairs.iterrows():
            stock_a, stock_b = row['Pair'][0], row['Pair'][1]
            
            # 【修正】Gatev (2006) 標準實作：交易期第一天需「重新歸一化」讓價差從 0 開始
            # 若不重置，前期的飄移會導致價差一直遠離 0，永遠無法觸發對稱的上下 2 個標準差進場線
            norm_price_a = trading_data[stock_a] / trading_data[stock_a].iloc[0]
            norm_price_b = trading_data[stock_b] / trading_data[stock_b].iloc[0]
            spread = norm_price_a - norm_price_b
            
            ret_a = trading_data[stock_a].pct_change().fillna(0).values
            ret_b = trading_data[stock_b].pct_change().fillna(0).values
            
            pair_params.append({
                'stock_a': stock_a, 'stock_b': stock_b, 'spread': spread.values,
                'ret_a': ret_a, 'ret_b': ret_b, 'w_a': 0.5, 'w_b': 0.5, 'sign_beta': 1, 
                'entry_threshold': self.entry * row['Std'], 'exit_threshold': self.exit,
                'pos': 0, 'stopped_out': False, 'cumulative_pnl': 0.0
            })
            
        for t in range(1, len(trading_data)):
            daily_mtm, daily_cost = 0.0, 0.0
            for p in pair_params:
                pos, pair_mtm = p['pos'], 0.0
                if pos == 1: 
                    pair_mtm = capital_per_pair * (p['w_a'] * p['ret_a'][t] - p['w_b'] * p['ret_b'][t])
                elif pos == -1: 
                    pair_mtm = capital_per_pair * (-p['w_a'] * p['ret_a'][t] + p['w_b'] * p['ret_b'][t])
                daily_mtm += pair_mtm
                
                if p['stopped_out']:
                    p['cumulative_pnl'] += pair_mtm
                    continue
                
                current_spread, new_pos = p['spread'][t], pos
                est_roi = (p['cumulative_pnl'] + pair_mtm) / capital_per_pair if capital_per_pair > 0 else 0
                
                if self.pair_max_loss_pct > 0 and est_roi <= -self.pair_max_loss_pct:
                    new_pos, p['stopped_out'] = 0, True
                elif pos == 1 and current_spread >= p['exit_threshold']: new_pos = 0
                elif pos == -1 and current_spread <= p['exit_threshold']: new_pos = 0
                elif new_pos == 0:
                    if current_spread > p['entry_threshold']: new_pos = -1
                    elif current_spread < -p['entry_threshold']: new_pos = 1
                        
                pair_cost = abs(new_pos - pos) * capital_per_pair * self.fee if new_pos != pos else 0.0
                daily_cost += pair_cost
                p['pos'] = new_pos
                p['cumulative_pnl'] += (pair_mtm - pair_cost)
                
            period_pnl[t] = daily_mtm - daily_cost
                
        print("\n  --- 視窗內各配對損益結算 ---")
        for p in pair_params:
            roi = (p['cumulative_pnl'] / capital_per_pair) * 100 if capital_per_pair > 0 else 0
            print(f"  配對 {p['stock_a']}-{p['stock_b']:<11} | 分配資金: ${capital_per_pair:,.0f} | 淨損益: ${p['cumulative_pnl']:>8.2f} | 區間報酬: {roi:>6.2f}%")
                
        return period_pnl

    def run_backtest(self):
        total_days = len(self.data)
        print(f"\n開始執行 [最小距離法 SSD] 配對回測... (總交易日: {total_days} 天)")
        window_capital = int(self.initial_capital // self.capital_tranches)
        portfolio_pnl = np.zeros(total_days)
        
        for start_idx in range(0, total_days - self.formation_days - self.trading_days + 1, self.rolling_window):
            form_end_idx = start_idx + self.formation_days
            trade_end_idx = form_end_idx + self.trading_days
            
            formation_data = self.data.iloc[start_idx:form_end_idx]
            trading_data = self.data.iloc[form_end_idx:trade_end_idx]
            
            best_pairs = self.find_best_pairs(formation_data)
            print(f"\n" + "="*50)
            print(f"新視窗期間: {trading_data.index[0].date()} 至 {trading_data.index[-1].date()}")
            if best_pairs.empty:
                print("  本期無顯著配對。")
            else:
                for _, row in best_pairs.iterrows():
                    print(f"  選定配對: {row['Pair'][0]:<5} & {row['Pair'][1]:<5} | SSD: {row['SSD']:.4f} | Std: {row['Std']:.4f}")
                record_df = best_pairs.copy()
                record_df['Window_Start'] = trading_data.index[0].date()
                record_df['Window_End'] = trading_data.index[-1].date()
                self.history_pairs.append(record_df)
                
            cohort_pnl = self.trade_pairs(trading_data, best_pairs, window_capital)
            portfolio_pnl[form_end_idx:trade_end_idx] += cohort_pnl
            
            window_total_pnl = cohort_pnl.sum()
            window_roi = (window_total_pnl / window_capital) * 100 if window_capital > 0 else 0
            print(f"  >>> 【視窗結算】 投入本金: ${window_capital:,.0f} | 總淨利: ${window_total_pnl:,.2f} | 總報酬: {window_roi:.2f}%")
            print("="*50)
            
        dates = self.data.index
        self.cumulative_returns = pd.Series(self.initial_capital + np.cumsum(portfolio_pnl), index=dates)
        self.portfolio_returns = self.cumulative_returns.pct_change().fillna(0)
        self._print_statistics()
        self._plot_results()

    def _print_statistics(self):
        annualized_return = self.portfolio_returns.mean() * 252
        annualized_vol = self.portfolio_returns.std() * np.sqrt(252)
        sharpe_ratio = annualized_return / annualized_vol if annualized_vol != 0 else 0
        running_max = self.cumulative_returns.cummax()
        drawdown = (self.cumulative_returns - running_max) / running_max
        max_drawdown = drawdown.min()
        final_value = self.cumulative_returns.iloc[-1] if not self.cumulative_returns.empty else self.initial_capital
        
        print("\n" + "="*40)
        print("【最小距離法 SSD - 回測績效統計】")
        print(f"年化報酬率: {annualized_return*100:.2f}%")
        print(f"年化波動率: {annualized_vol*100:.2f}%")
        print(f"夏普值 (Sharpe): {sharpe_ratio:.2f}")
        print(f"最大回撤 (MDD): {max_drawdown*100:.2f}%")
        print(f"期末淨值: ${final_value:,.2f}")
        print("="*40)

    def _plot_results(self):
        running_max = self.cumulative_returns.cummax()
        drawdown = (self.cumulative_returns - running_max) / running_max
        rolling_return = self.portfolio_returns.rolling(window=126).mean() * 252
        rolling_vol = self.portfolio_returns.rolling(window=126).std() * np.sqrt(252)
        rolling_sharpe = rolling_return / rolling_vol.replace(0, np.nan).fillna(0)

        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [2, 1, 1.2]})
        ax1.plot(self.cumulative_returns.index, self.cumulative_returns.values, label='Portfolio Value', color='darkorange')
        ax1.set_title('Cumulative Portfolio Value (SSD Strategy)', fontsize=14)
        ax1.grid(True, linestyle='--', alpha=0.6)

        ax2.fill_between(drawdown.index, drawdown.values * 100, 0, color='crimson', alpha=0.5, label='Drawdown (%)')
        ax2.set_title('Maximum Drawdown (%)', fontsize=12)
        ax2.grid(True, linestyle='--', alpha=0.6)

        ax3.plot(rolling_sharpe.index, rolling_sharpe.values, color='forestgreen', label='126-Day Rolling Sharpe')
        ax3.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
        ax3.set_title('Rolling Annualized Sharpe Ratio (126-Day)', fontsize=12)
        ax3.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()

    def export_data(self, save_dir=r'..\results', prefix='ssd_'):
        os.makedirs(save_dir, exist_ok=True)
        pd.DataFrame({'Daily_Return': self.portfolio_returns, 'Cumulative_Value': self.cumulative_returns}).to_csv(os.path.join(save_dir, f'{prefix}portfolio_nav.csv'))
        if self.history_pairs:
            pairs_df = pd.concat(self.history_pairs, ignore_index=True)
            pairs_df['Stock_A'] = pairs_df['Pair'].apply(lambda x: x[0])
            pairs_df['Stock_B'] = pairs_df['Pair'].apply(lambda x: x[1])
            pairs_df[['Window_Start', 'Window_End', 'Sector', 'Stock_A', 'Stock_B', 'SSD', 'Std']].to_csv(os.path.join(save_dir, f'{prefix}traded_pairs.csv'), index=False)
        self.data.to_pickle(os.path.join(save_dir, f'{prefix}price_pivot.pkl'))
        with open(os.path.join(save_dir, f'{prefix}sector_map.json'), 'w', encoding='utf-8') as f:
            json.dump(self.sector_map, f, ensure_ascii=False, indent=4)
        print(f"\n【資料匯出】 已儲存 {prefix} 系列檔案至 {save_dir}")

In [ ]:
# 主程式執行區
# ==========================================
if __name__ == "__main__":
    # 使用 FetchData 類別進行統一調用
    data_fetcher = FetchData(
        db_path=DB_PATH,
        start_date=START_DATE,
        end_date=END_DATE,
        target_sector=TARGET_SECTOR,
        use_dynamic=USE_DYNAMIC_SECTORS,
        imputed_path=IMPUTED_SECTOR_PATH,
        min_days=MIN_HISTORY_DAYS
    )
    
    price_pivot, sector_map = data_fetcher.fetch_all()
    
    print(f'\n【行業分佈】\n{pd.Series(sector_map).value_counts().head(10)}\n')

    # ==========================================
    # 策略切換區域 (您可自由切換欲執行的策略)
    # ==========================================
    
    # 【執行 SSD 最小距離法策略】
    backtester = SSDPairsTrading(
        data=price_pivot, 
        sector_map=sector_map,
        formation_days=FORMATION_WINDOW,
        trading_days=TRADING_WINDOW,
        rolling_window=ROLLING_WINDOW,
        capital_tranches=CAPITAL_TRANCHES,
        initial_capital=INITIAL_CAPITAL,
        z_entry=Z_ENTRY,
        z_exit=Z_EXIT,
        top_n_pairs=TOP_N_PAIRS,
        fee=TRANSACTION_COST,
        pair_max_loss_pct=PAIR_MAX_LOSS_PCT
    )

    # 執行回測
    backtester.run_backtest()
    
    # 執行數據匯出 (會自動以 "ssd_" 為檔名開頭儲存)
    backtester.export_data()